In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt


In [2]:
def selectkbest(indep_X,dep_Y,n):
    
        test = SelectKBest(score_func=chi2, k=n)
        fit1= test.fit(indep_X,dep_Y)
        # summarize scores       
        selectk_features = fit1.transform(indep_X)
        return selectk_features
    

# SelectKBest - selects only important features
# Parameters - indep_X : input features , dep_y :output /target variable , n: number of features
# chi2 - checks relationship between all input features and target ,keeps top n features
# fit - calculates score for each selected feature and ranks them
# keeps only selected best features ,removes weak columns , returns reduced dataset
# returns top n features dataset

In [3]:
def split_scalar(indep_X,dep_Y):
    
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test
    

# split_scalar - splits data train/test ; standardises the features
# train_test_split :splits data for training set ,test set ; 0.25 - 25% test size , 75% training ;random_state=0 - same split everytime
# sc - scales and standardises data so that all features are treated equally; mean =0 std =1
# sc.fit_transform - learns mean , std and applies on training data
# sc.transform - apply same mean std and scale test data
# return - returns scaled training , test features

In [4]:
def r2_prediction(regressor,X_test,y_test):
    
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2

# Calculates how well the trained model predicts test data using the R² score.
# y_pred : passing the input test values to the trained models to predict output test values 
# r2 : imports r2 to evaluate regression models
# calculates difference between actual / predicted test values and returns r2

In [5]:
def Linear(X_train,y_train,X_test):    
    
        from sklearn.linear_model import LinearRegression
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   

# Training Linear Regression Model
# import LR model and creaters model object "Regressor"
# regressor.fit : model is created and learns relationship btw training data (input and output features)
# r2_prediction : uses function predicts y_pred and calculates r2 score

In [6]:
def svm_linear(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  

# Training Support Vector Regression - Linear Model
# kernel : linear ( tries to fit a straight line)
# fit : Model is created
# r2 : predicts y_pred and calculates r2 score

In [7]:
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'rbf')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  

# Training Support Vector Regression - Non Linear Model
# Kernel : RBF Radial Basis Function - Learns non linear patterns and fits curved relationships

In [8]:
def Decision(X_train,y_train,X_test):
            
        from sklearn.tree import DecisionTreeRegressor
        regressor = DecisionTreeRegressor(random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  

# Training Decision Tree Regressor Model
# random_state = 0 ; ensures same result in every run (reproduceability)
# .fit - model keeps splitting until it learns patterns
# Predicts and calculates r2 score

In [9]:
def random(X_train,y_train,X_test):       
       
        from sklearn.ensemble import RandomForestRegressor
        regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 

In [10]:
# Training Random Forest Regressor Model
# n_estimators = 10 , 10 decision trees
# .fit : Each tree learns pattern independently
# each tree predicts calculates average of all the trees and then calculates r2

In [11]:
def selectk_regression(acclin,accsvml,accsvmnl,accdes,accrf): 

    dataframe = pd.DataFrame(columns=['Linear', 'SVMl', 'SVMnl', 'Decision', 'Random'])

    dataframe.loc['ChiSquare'] = [
                                    acclin[0],
                                    accsvml[0],
                                    accsvmnl[0],
                                    accdes[0],
                                    accrf[0]
                                    ]
    
    for number,idex in enumerate(dataframe.index):
        dataframe.loc[idex, 'Linear'] = acclin[number]
        dataframe.loc[idex, 'SVML'] = accsvml[number]     
        dataframe.loc[idex, 'SVMnl'] = accsvmnl[number]
        dataframe.loc[idex, 'Decision'] = accdes[number]
        dataframe.loc[idex, 'Random'] = accrf[number]
        
    return dataframe
    

# This function creates a comparison table of different ML models’ R² scores
# pd.dataframe : creates column names
# .loc['ChiSquare']: adds 1 row  and stores all model accuracies in that row


In [30]:
dataset1=pd.read_csv("prep.csv",index_col=None)

df2=dataset1

df2 = pd.get_dummies(df2, drop_first=True)

indep_X = df2.drop('classification_yes', axis=1)
dep_Y = df2['classification_yes']


kbest = selectkbest(indep_X,dep_Y,7)      

acclin=[]
accsvml=[]
accsvmnl=[]
accdes=[]
accrf=[]

# This code prepares data (encoding + splitting), selects best features, and initializes storage lists for comparing multiple ML models

In [31]:
X_train, X_test, y_train, y_test=split_scalar(kbest,dep_Y)  

  
r2_lin=Linear(X_train,y_train,X_test)
acclin.append(r2_lin)
    
r2_sl=svm_linear(X_train,y_train,X_test)    
accsvml.append(r2_sl)
    
r2_NL=svm_NL(X_train,y_train,X_test)
accsvmnl.append(r2_NL)
    
r2_d=Decision(X_train,y_train,X_test)
accdes.append(r2_d)
    
r2_r=random(X_train,y_train,X_test)
accrf.append(r2_r)
    
    
result=selectk_regression(acclin,accsvml,accsvmnl,accdes,accrf)


# Model Evaluation Loop 
# Splits training , test data
# Evaluates each model and stores the accuracy value respectively

In [20]:
result
#k=3

,Linear,SVMl,SVMnl,Decision,Random,SVML
ChiSquare,0.287968,0.255063,0.333342,0.262153,0.528212,0.255063


In [23]:
result
#k=4

,Linear,SVMl,SVMnl,Decision,Random,SVML
ChiSquare,0.304963,0.256858,0.430949,0.479167,0.599392,0.256858


In [26]:
result
#k=5

,Linear,SVMl,SVMnl,Decision,Random,SVML
ChiSquare,0.551985,0.545395,0.749654,0.696181,0.836806,0.545395


In [29]:
result
#k=6

,Linear,SVMl,SVMnl,Decision,Random,SVML
ChiSquare,0.599041,0.586446,0.838987,0.869792,0.897569,0.586446


In [32]:
result
#k=7

,Linear,SVMl,SVMnl,Decision,Random,SVML
ChiSquare,0.657035,0.641906,0.893007,0.826389,0.916233,0.641906


In [19]:
#### k=7 gives the overall accuracy for all the models 